# CampusShield A1 — Colab T4 trainer (4-class, real + mock)

Trains the **A1 RF classifier** on the 170 GB Kaggle dataset
`sgluege/noisy-drone-rf-signal-classification-v2` **without ever downloading it whole**.

- **Stream**: download a batch of `.pt` files -> extract the 735 features -> delete raw -> repeat.
  170 GB never has to fit on Colab's ~80 GB disk.
- **real** `drone` + `noise` come from Kaggle; **mock** `wifi` + `bluetooth` are synthesized with
  your repo's exact `rf_synth`, so the model stays **4-class** and drops into `infer.py` unchanged.
- Output: `a1_rf_clf.pkl` + `a1_rf_scaler.pkl` (+ `a1_meta.json`). Copy the two `.pkl`s into `models/`.

> Runtime -> Change runtime type -> **T4 GPU** before running.

## Cell 1 - setup

In [ ]:
!pip -q install kaggle pywavelets
import torch, xgboost
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())  # want True
print("xgboost", xgboost.__version__)

## Cell 2 - Kaggle auth (upload kaggle.json)

In [ ]:
from google.colab import files
import os
print("Upload kaggle.json  (kaggle.com -> Account -> Create New API Token)")
up = files.upload()
os.makedirs("/root/.kaggle", exist_ok=True)
open("/root/.kaggle/kaggle.json", "wb").write(list(up.values())[0])
os.chmod("/root/.kaggle/kaggle.json", 0o600)
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
print("Kaggle auth OK")

## Cell 3 - your exact feature extractor + 8-bit quantization

In [ ]:
import numpy as np, pywt
from scipy import signal as sp
SAMPLE_RATE = 2.4e6
N_IQ = 65536                                   # IQ_SAMPLES_PER_FILE
RF_CLASS_NAMES = ["noise", "drone", "wifi", "bluetooth"]

def _feature_names():
    n = [f"fft_{i}" for i in range(512)] + [f"psd_{i}" for i in range(128)]
    n += ["mean","std","max","p25","p75","rms","occupancy","entropy"]
    for b in ["cA4","cD4","cD3","cD2","cD1"]:
        n += [f"wav_{b}_mean", f"wav_{b}_std", f"wav_{b}_energy", f"wav_{b}_max"]
    n += ["inst_mean","inst_std","inst_max"] + [f"ac_{i}" for i in range(64)]
    return n
N_FEATURES = len(_feature_names())             # 735

def extract_features(s):
    s = np.asarray(s, dtype=np.complex64)
    if s.size < 4096: s = np.pad(s, (0, 4096 - s.size))
    s = s[:65536]; mag = np.abs(s)
    f = np.abs(np.fft.fft(s[:1024]))[:512]; fft = f / (np.max(f) + 1e-10)
    _, psd = sp.welch(mag, fs=SAMPLE_RATE, nperseg=256); psd = psd[:128]
    psd = psd / (np.max(psd) + 1e-10)
    if psd.size < 128: psd = np.pad(psd, (0, 128 - psd.size))
    p = mag / (np.sum(mag) + 1e-10); ent = -np.sum(p * np.log2(p + 1e-10))
    stats = np.array([np.mean(mag), np.std(mag), np.max(mag), np.percentile(mag,25),
                      np.percentile(mag,75), np.mean(mag**2),
                      np.sum(mag > np.mean(mag)) / len(mag), ent])
    wf = []
    for c in pywt.wavedec(mag[:1024], "db4", level=4):
        wf += [np.mean(c), np.std(c), np.mean(c**2), np.max(np.abs(c))]
    inst = np.diff(np.unwrap(np.angle(s[:4096])))
    ff = np.array([np.mean(inst), np.std(inst), np.max(np.abs(inst))])
    full = np.correlate(mag[:512], mag[:512], "full"); c0 = full[full.size//2]
    ac = full[full.size//2:][:64] / (c0 + 1e-10)
    if ac.size < 64: ac = np.pad(ac, (0, 64 - ac.size))
    feats = np.concatenate([fft, psd, stats, np.array(wf), ff, ac]).astype(np.float32)
    return np.nan_to_num(feats)

def iq_roundtrip(s):
    # replicate save_iq -> load_iq 8-bit quantization so Colab-train == Pi-infer
    s = np.asarray(s, dtype=np.complex64)
    i = np.clip(np.real(s) * 127.0 + 127.5, 0, 255).astype(np.uint8)
    q = np.clip(np.imag(s) * 127.0 + 127.5, 0, 255).astype(np.uint8)
    return (((i.astype(np.float32) - 127.5) / 127.5)
            + 1j * ((q.astype(np.float32) - 127.5) / 127.5)).astype(np.complex64)

def pt_to_features(d):
    iq = np.asarray(d["x_iq"])
    s = (iq[0, :] + 1j * iq[1, :]).astype(np.complex64)[:65536]
    s = s / (np.max(np.abs(s)) + 1e-9)          # real IQ: normalize like fetch_rf
    return extract_features(iq_roundtrip(s))

## Cell 4 - mock wifi/bluetooth synth (verbatim from your rf_synth)

In [ ]:
# Same signatures your pipeline trains on today; used until you record real wifi/bt.
def _awgn(n, sigma, rng):
    return (rng.normal(0, sigma, n) + 1j * rng.normal(0, sigma, n)).astype(np.complex64)
def _tone(n, f, rng, phase=None):
    t = np.arange(n); ph = rng.uniform(0, 2*np.pi) if phase is None else phase
    return np.exp(1j * (2*np.pi*f*t + ph)).astype(np.complex64)
def _ofdm(n, bw, rng):
    sig = np.zeros(n, dtype=np.complex64); nc = max(4, int(bw*64))
    for fc in rng.uniform(-bw/2, bw/2, nc):
        sig += _tone(n, fc, rng) * rng.uniform(0.3, 1.0)
    return sig / np.sqrt(nc)
def _hop(n, dwell, bw, rng, span=0.4):
    sig = np.zeros(n, dtype=np.complex64); i = 0
    while i < n:
        seg = min(dwell, n - i); fc = rng.uniform(-span, span); t = np.arange(seg)
        sig[i:i+seg] = (_ofdm(seg, bw, rng) * np.exp(1j*2*np.pi*fc*t)).astype(np.complex64)
        i += seg
    return sig
def _burst(sig, period, duty):
    env = np.zeros(len(sig), dtype=np.float32); i = 0
    while i < len(sig):
        env[i:i+int(period*duty)] = 1.0; i += period
    return sig * env
def synth(kind, rng, snr_db=12.0):
    ns = 0.08; sig = _awgn(N_IQ, ns, rng)
    if kind == "noise": return sig
    amp = 10 ** (snr_db/20.0) * ns
    if kind == "wifi":       body = _burst(_ofdm(N_IQ, 0.7, rng), 8000, 0.35)
    elif kind == "bluetooth":body = _hop(N_IQ, 900, 0.04, rng, 0.45)
    else: raise ValueError(kind)
    body = body / (np.std(body) + 1e-9)
    return (sig + amp * body).astype(np.complex64)

## Cell 5 - mount Drive (resumable; Colab disconnects ~12h)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT = "/content/drive/MyDrive/campusshield_rf"; os.makedirs(CKPT, exist_ok=True)

## Cell 6 - index files + class map

In [ ]:
SLUG = "sgluege/noisy-drone-rf-signal-classification-v2"
listing = api.dataset_list_files(SLUG).files
names = [f.name for f in listing]
print("files listed:", len(names))
pt_names = [n for n in names
            if os.path.basename(n).startswith("IQdata_sample") and n.endswith(".pt")]
print("IQ .pt files found:", len(pt_names))

# small csv maps class index -> name (so we can split noise vs drone)
import glob, zipfile, csv
class_names = None
cs = [n for n in names if n.endswith("class_stats.csv")]
if cs:
    api.dataset_download_file(SLUG, cs[0], path="/content", force=True)
    for z in glob.glob("/content/*.zip"): zipfile.ZipFile(z).extractall("/content")
    fp = glob.glob("/content/**/class_stats.csv", recursive=True)
    if fp:
        rows = list(csv.DictReader(open(fp[0])))
        col = "class" if "class" in rows[0] else list(rows[0])[0]
        class_names = [r[col] for r in rows]; print("classes:", class_names)

# If pt_names == 0 the listing was truncated (huge dataset). Inspect and adapt:
if not pt_names:
    print("!! no .pt found via listing. First 20 names:", names[:20])

## Cell 7 - stream REAL drone/noise: download -> featurize -> discard

In [ ]:
import concurrent.futures as cf, tempfile, shutil, time
TARGET_PER_CLASS = 10_000_000     # "as many as possible" - stops when files run out
WORKERS = 8                        # concurrent downloads (network-bound)

Xp, yp = os.path.join(CKPT, "X.npy"), os.path.join(CKPT, "y.npy")
done_p = os.path.join(CKPT, "done.txt")
X = list(np.load(Xp)) if os.path.exists(Xp) else []
y = list(np.load(yp)) if os.path.exists(yp) else []
done = set(open(done_p).read().split()) if os.path.exists(done_p) else set()
print(f"resume: {len(y)} samples already, {len(done)} files done")

def to_label(yi):
    nm = class_names[yi] if class_names and yi < len(class_names) else str(yi)
    return 0 if str(nm).lower().startswith("noise") else 1   # noise=0, drone=1

def fetch_one(name):
    tmp = tempfile.mkdtemp()
    try:
        api.dataset_download_file(SLUG, name, path=tmp, force=True, quiet=True)
        got = [os.path.join(r, fn) for r, _, fs in os.walk(tmp) for fn in fs
               if fn.endswith((".pt", ".zip"))][0]
        if got.endswith(".zip"):
            zipfile.ZipFile(got).extractall(tmp)
            got = [os.path.join(r, fn) for r, _, fs in os.walk(tmp) for fn in fs
                   if fn.endswith(".pt")][0]
        d = torch.load(got, map_location="cpu", weights_only=False)
        yi = int(np.asarray(d["y"]).ravel()[0])
        return name, pt_to_features(d), to_label(yi)
    except Exception:
        return name, None, None
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

todo = [n for n in pt_names if n not in done]
t0 = time.time()
with cf.ThreadPoolExecutor(max_workers=WORKERS) as ex:
    for i, (name, feat, lab) in enumerate(ex.map(fetch_one, todo), 1):
        done.add(name)
        if feat is not None:
            X.append(feat); y.append(lab)
        if i % 500 == 0:
            np.save(Xp, np.array(X)); np.save(yp, np.array(y))
            open(done_p, "w").write("\n".join(done))
            print(f"{i}/{len(todo)}  kept={len(y)}  {i/(time.time()-t0):.1f} files/s")
np.save(Xp, np.array(X)); np.save(yp, np.array(y)); open(done_p, "w").write("\n".join(done))
X = np.array(X); y = np.array(y)
print("REAL done:", X.shape, " counts:", np.bincount(y, minlength=2))

## Cell 8 - generate MOCK wifi/bluetooth (4-class fill-in)

In [ ]:
# Raise to roughly match your real per-class count for balance; 8000 is a fine start.
N_MOCK_PER_CLASS = 8000
rng = np.random.default_rng(42)
Xm, ym = [], []
for kind, idx in [("wifi", 2), ("bluetooth", 3)]:
    for k in range(N_MOCK_PER_CLASS):
        s = synth(kind, rng, snr_db=float(rng.uniform(8, 16)))
        Xm.append(extract_features(iq_roundtrip(s))); ym.append(idx)
        if (k + 1) % 2000 == 0: print(kind, k + 1)
Xm = np.array(Xm); ym = np.array(ym)
np.save(os.path.join(CKPT, "Xmock.npy"), Xm); np.save(os.path.join(CKPT, "ymock.npy"), ym)
print("MOCK done:", Xm.shape, " counts:", np.bincount(ym, minlength=4))

## Cell 9 - combine + train GPU XGBoost (4-class)

In [ ]:
import joblib, json
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb

Xall = np.vstack([X, Xm]); yall = np.concatenate([y, ym])
print("combined:", Xall.shape, " counts:", np.bincount(yall, minlength=4),
      "->", RF_CLASS_NAMES)

sc = StandardScaler(); Xs = sc.fit_transform(Xall)
Xtr, Xte, ytr, yte = train_test_split(Xs, yall, test_size=0.2, stratify=yall,
                                       random_state=42)
clf = xgb.XGBClassifier(n_estimators=600, max_depth=8, learning_rate=0.1,
                        tree_method="hist", device="cuda",           # T4 GPU
                        eval_metric="mlogloss", n_jobs=-1, random_state=42)
clf.fit(Xtr, ytr)
print("test acc:", round(clf.score(Xte, yte), 4))
print(classification_report(yte, clf.predict(Xte), target_names=RF_CLASS_NAMES))
print(confusion_matrix(yte, clf.predict(Xte)))

## Cell 10 - save drop-in artifacts + download

In [ ]:
joblib.dump(clf, f"{CKPT}/a1_rf_clf.pkl")
joblib.dump(sc,  f"{CKPT}/a1_rf_scaler.pkl")
json.dump({"best_model": "XGBoost", "classes": RF_CLASS_NAMES,
           "n_features": int(Xall.shape[1]),
           "counts": {n: int(c) for n, c in zip(RF_CLASS_NAMES,
                                                np.bincount(yall, minlength=4))}},
          open(f"{CKPT}/a1_meta.json", "w"), indent=2)
from google.colab import files
for fn in ["a1_rf_clf.pkl", "a1_rf_scaler.pkl", "a1_meta.json"]:
    files.download(f"{CKPT}/{fn}")

## Drop-in on the Pi / laptop (no code change)

1. Copy `a1_rf_clf.pkl` **and** `a1_rf_scaler.pkl` into your repo's `models/` folder
   (both are required - the scaler must match the model).
2. Test:
   ```
   python -m ml.a1_rf_classifier.infer dataset/drone/drone_0000.bin
   python -m ml.runtime.live_detector --once --no-buzzer --no-telegram
   ```
`infer.py` reads them from `models/` automatically.

**Fragile spot:** if Cell 6 finds 0 `.pt` files, `dataset_list_files` was truncated for this
huge dataset - paste what `names[:20]` prints and adapt the filter, or run this as a
**Kaggle Notebook** (data pre-mounted, `os.walk` sees every file, zero download).